# Computer Networks and HTTP

Notebook for **Chapter 4** of the *cs* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/cs/ch04.html

Tip: change a number, predict the output, then run the cell again.

## 4.1 Layers, IP Addresses and DNS

In [ ]:
import ipaddress
net = ipaddress.ip_network("192.168.10.0/24")
print("network:", net.network_address, "mask:", net.netmask, "broadcast:", net.broadcast_address)
print("usable hosts:", net.num_addresses - 2)
print("10.0.5.77 in 10.0.0.0/16?", ipaddress.ip_address("10.0.5.77") in ipaddress.ip_network("10.0.0.0/16"))
print("subnets of /24 into /26:", [str(s) for s in net.subnets(new_prefix=26)])
print("private?", ipaddress.ip_address("192.168.1.5").is_private, ipaddress.ip_address("8.8.8.8").is_private)
print("bits for /20:", 32-20, "hosts:", 2**(32-20)-2)

In [ ]:
cache={}; lookups=0
def resolve(name,t,ttl=60):
    global lookups
    if name in cache and cache[name]>t: return "cache"
    lookups+=1; cache[name]=t+ttl; return "network"
print([resolve("api.example.com",t) for t in (0,10,59,61,70)], "network lookups:",lookups)

['network', 'cache', 'cache', 'network', 'cache'] network lookups: 2


## 4.2 TCP versus UDP

In [ ]:
import socket, threading
srv = socket.socket(); srv.bind(("127.0.0.1", 0)); srv.listen()
port = srv.getsockname()[1]
def serve():
    c, _ = srv.accept()
    data = c.recv(1024); c.sendall(b"ECHO:" + data); c.close()
threading.Thread(target=serve, daemon=True).start()
c = socket.create_connection(("127.0.0.1", port))   # the handshake happens here
c.sendall(b"hello"); print("TCP reply:", c.recv(1024)); c.close()

u = socket.socket(socket.AF_INET, socket.SOCK_DGRAM); u.bind(("127.0.0.1", 0))
s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
s.sendto(b"ping", ("127.0.0.1", u.getsockname()[1]))
print("UDP got:", u.recvfrom(1024)[0])

TCP reply: b'ECHO:hello'
UDP got: b'ping'


## 4.3 HTTP, TLS and Other Application Protocols

In [ ]:
c=socket.create_connection(("127.0.0.1",hp))
c.sendall(b"GET /login HTTP/1.1\r\nHost: localhost\r\n\r\n")

## 4.4 Load Balancers, Proxies, CDNs and Firewalls

In [ ]:
costs=[9,1,1,9,1,1,9,1,1]
rr=[0,0,0]
for i,c in enumerate(costs): rr[i%3]+=c
lc=[0,0,0]
for c in costs: lc[lc.index(min(lc))]+=c
print("round robin load:",rr,"busiest:",max(rr)); print("least loaded    :",lc,"busiest:",max(lc))

round robin load: [27, 3, 3] busiest: 27
least loaded    : [11, 10, 12] busiest: 12


## 4.5 Latency, Bandwidth and Throughput

In [ ]:
def transfer_ms(size_bytes, rtt_ms, mbps, round_trips=1):
    return round_trips*rtt_ms + size_bytes*8/(mbps*1e6)*1000
for name,size in [("1 KB API reply",1_000),("1 MB image",1_000_000),("1 GB dataset",1_000_000_000)]:
    print(f"{name:15s} RTT=50ms: 10Mbps={transfer_ms(size,50,10):10.1f} ms   1000Mbps={transfer_ms(size,50,1000):9.1f} ms")
print("BDP at 100 Mbps, 80 ms RTT:", 100e6*0.080/8/1e3, "KB in flight")

## 4.6 TCP Congestion Control and the Cost of a Cold Connection

In [ ]:
import math
size, mss, rtt = 1_000_000, 1460, 0.050
need = math.ceil(size / mss)
cwnd, sent, rounds = 10, 0, 0
while sent < need:
    sent += cwnd
    rounds += 1
    print(f"round {rounds}: cwnd={cwnd:4d} total sent={min(sent, need)}")
    cwnd *= 2
print("segments:", need, "rounds:", rounds, "time ms:", int(rounds * rtt * 1000))

round 1: cwnd=  10 total sent=10
round 2: cwnd=  20 total sent=30
round 3: cwnd=  40 total sent=70
round 4: cwnd=  80 total sent=150
round 5: cwnd= 160 total sent=310
round 6: cwnd= 320 total sent=630
round 7: cwnd= 640 total sent=685
segments: 685 rounds: 7 time ms: 350


In [ ]:
bw_bits, rtt = 100_000_000, 0.080
bdp = bw_bits / 8 * rtt
print("BDP bytes:", int(bdp))
thr = 65536 / rtt
print("64 KiB window throughput Mbit/s:", round(thr * 8 / 1e6, 1), "share of link:", round(thr * 8 / bw_bits * 100, 1), "%")

BDP bytes: 1000000
64 KiB window throughput Mbit/s: 6.6 share of link: 6.6 %


## 4.7 TLS 1.3, HTTP/2 and HTTP/3

## Chapter recap

## Practice set

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/cs/ch04.html